# F2 threshold tuning for multilabel toxic comment classifiers

This notebook runs one forward pass per model on the saved dev Arrow dataset, caches probabilities, and tunes one threshold per label from `0.30` to `0.80` to maximize label-level F2, which weights recall more than precision. Recall, precision, and closeness to `0.50` are used as tie-breakers.


In [ ]:
import gc
import inspect
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from accelerate import Accelerator
from datasets import load_from_disk
from sklearn.metrics import (
    classification_report,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, Subset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
)


In [ ]:
MODEL_PATHS = {
    "DistilBERT": {
        "dev_data_path": "/kaggle/input/datasets/mohamedmouldibziou/devdatadistilbert-txcc",
        "model_to_import_path": "/kaggle/input/notebooks/mohamedmouldibziou/fork-of-distilbert-base-uncased-finetuning-4-epoch/best_model_distilbert-base-uncased",
    },
    "HateBERT": {
        "dev_data_path": "/kaggle/input/datasets/mohamedmouldibziou/devdatahatebert-txcc",
        "model_to_import_path": "/kaggle/input/notebooks/mohamedmouldibziou/hatebert-4th-epoch/best_model_HateBERT",
    },
    "RoBERTaBase": {
        "dev_data_path": "/kaggle/input/datasets/mohamedmouldibziou/devdatarobertabase-txcc",
        "model_to_import_path": "/kaggle/input/notebooks/mohamedmouldibziou/fork-of-roberta-base-finetuning-4-epochs/best_model_roberta-base",
    },
}

LABEL_COLS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]

THRESHOLD_GRID = np.round(np.arange(0.30, 0.801, 0.01), 2)
BASELINE_THRESHOLD = 0.50
BATCH_SIZE = 32
NUM_WORKERS = 4

OUTPUT_DIR = Path("/kaggle/working/threshold_tuning_f2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count() if torch.cuda.is_available() else 0}")
print(f"Threshold candidates: {len(THRESHOLD_GRID)} values from {THRESHOLD_GRID[0]} to {THRESHOLD_GRID[-1]}")


In [ ]:
def load_dev_dataset(dev_data_path):
    """Load a saved HF Dataset and keep only model inputs plus labels."""
    dataset = load_from_disk(dev_data_path)
    columns = ["input_ids", "attention_mask", "labels"]
    if "token_type_ids" in dataset.column_names:
        columns.append("token_type_ids")

    missing = sorted(set(columns) - set(dataset.column_names))
    if missing:
        raise ValueError(f"Dataset at {dev_data_path} is missing columns: {missing}")

    dataset.set_format(type="torch", columns=columns)
    return dataset, columns


def model_accepts_token_type_ids(model):
    """Avoid passing token_type_ids to models such as RoBERTa that do not use them."""
    forward_params = inspect.signature(model.forward).parameters
    return "token_type_ids" in forward_params


def make_dataloader(dataset, tokenizer, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    worker_kwargs = {}
    if num_workers > 0:
        worker_kwargs = {
            "num_workers": num_workers,
            "pin_memory": torch.cuda.is_available(),
            "prefetch_factor": 2,
            "persistent_workers": True,
        }

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=data_collator,
        **worker_kwargs,
    )


def run_forward_pass(model_path, dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    """Run one optimized inference pass and return probabilities, labels, and avg loss."""
    accelerator = Accelerator(mixed_precision="fp16")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    dataloader = make_dataloader(dataset, tokenizer, batch_size, num_workers)

    accepts_token_type_ids = model_accepts_token_type_ids(model)
    model, dataloader = accelerator.prepare(model, dataloader)
    model.eval()

    loss_fn = torch.nn.BCEWithLogitsLoss(reduction="sum")
    gathered_logits = []
    gathered_labels = []
    total_loss = torch.tensor(0.0, device=accelerator.device)
    total_examples = torch.tensor(0, device=accelerator.device)

    with torch.no_grad():
        for batch in dataloader:
            labels = batch["labels"].float()
            model_inputs = {
                "input_ids": batch["input_ids"],
                "attention_mask": batch["attention_mask"],
            }
            if accepts_token_type_ids and "token_type_ids" in batch:
                model_inputs["token_type_ids"] = batch["token_type_ids"]

            outputs = model(**model_inputs)
            logits = outputs.logits
            loss = loss_fn(logits, labels)

            logits, labels = accelerator.gather_for_metrics((logits, labels))
            gathered_logits.append(logits.detach().float().cpu())
            gathered_labels.append(labels.detach().float().cpu())

            batch_examples = torch.tensor(labels.shape[0], device=accelerator.device)
            total_loss += accelerator.gather(loss.detach()).sum()
            total_examples += accelerator.gather(batch_examples).sum()

    all_logits = torch.cat(gathered_logits, dim=0).numpy()
    all_labels = torch.cat(gathered_labels, dim=0).numpy().astype(int)
    all_probs = 1.0 / (1.0 + np.exp(-all_logits))
    avg_loss = (total_loss / total_examples).item()

    accelerator.wait_for_everyone()
    accelerator.free_memory()
    del model, tokenizer, dataloader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return all_probs, all_labels, avg_loss


In [ ]:
def metric_dict(labels, preds, loss=None):
    metrics = {
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f2_macro": fbeta_score(labels, preds, beta=2, average="macro", zero_division=0),
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f2_micro": fbeta_score(labels, preds, beta=2, average="micro", zero_division=0),
    }
    if loss is not None:
        metrics["loss"] = float(loss)
    return metrics


def score_single_label(labels, probs, label_idx, threshold):
    label_true = labels[:, label_idx]
    label_pred = (probs[:, label_idx] >= threshold).astype(int)
    return {
        "precision": precision_score(label_true, label_pred, zero_division=0),
        "recall": recall_score(label_true, label_pred, zero_division=0),
        "f1": f1_score(label_true, label_pred, zero_division=0),
        "f2": fbeta_score(label_true, label_pred, beta=2, zero_division=0),
        "support": int(label_true.sum()),
        "predicted_positive": int(label_pred.sum()),
    }


def tune_thresholds(probs, labels, threshold_grid=THRESHOLD_GRID):
    rows = []
    best_rows = []

    for label_idx, label_name in enumerate(LABEL_COLS):
        label_rows = []
        for threshold in threshold_grid:
            scores = score_single_label(labels, probs, label_idx, threshold)
            row = {
                "label": label_name,
                "label_idx": label_idx,
                "threshold": float(threshold),
                **scores,
            }
            rows.append(row)
            label_rows.append(row)

        label_df = pd.DataFrame(label_rows)
        label_df["distance_from_0_5"] = (label_df["threshold"] - BASELINE_THRESHOLD).abs()
        best_row = label_df.sort_values(
            ["f2", "recall", "precision", "distance_from_0_5"],
            ascending=[False, False, False, True],
        ).iloc[0]
        best_rows.append(best_row.drop(labels=["distance_from_0_5"]).to_dict())

    sweep_df = pd.DataFrame(rows)
    best_df = pd.DataFrame(best_rows)
    threshold_vector = best_df.sort_values("label_idx")["threshold"].to_numpy(dtype=float)
    return sweep_df, best_df, threshold_vector


def evaluate_threshold_vector(probs, labels, thresholds, loss=None):
    thresholds = np.asarray(thresholds, dtype=float)
    preds = (probs >= thresholds.reshape(1, -1)).astype(int)
    return preds, metric_dict(labels, preds, loss=loss)


def save_json(path, payload):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


## Quick sanity check

This runs a small subset through one model to confirm the tensor shapes before the full threshold sweep. Set `RUN_SANITY_CHECK = False` if you want to skip it on Kaggle.


In [ ]:
RUN_SANITY_CHECK = True
SANITY_MODEL_NAME = "DistilBERT"
SANITY_ROWS = 128

if RUN_SANITY_CHECK:
    paths = MODEL_PATHS[SANITY_MODEL_NAME]
    sanity_dataset, sanity_columns = load_dev_dataset(paths["dev_data_path"])
    sanity_subset = Subset(sanity_dataset, range(min(SANITY_ROWS, len(sanity_dataset))))
    sanity_probs, sanity_labels, sanity_loss = run_forward_pass(
        paths["model_to_import_path"],
        sanity_subset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )

    print(f"Sanity model: {SANITY_MODEL_NAME}")
    print(f"Dataset columns used: {sanity_columns}")
    print(f"Probabilities shape: {sanity_probs.shape}")
    print(f"Labels shape: {sanity_labels.shape}")
    print(f"Average loss: {sanity_loss:.4f}")

    assert sanity_probs.shape == sanity_labels.shape
    assert sanity_probs.shape[1] == len(LABEL_COLS)

    del sanity_dataset, sanity_subset, sanity_probs, sanity_labels
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Run full threshold tuning

For each model this cell saves:

- `threshold_sweep.csv`: every label/threshold score.
- `best_thresholds.json`: best threshold vector, baseline metrics, tuned metrics, and per-label reports.


In [ ]:
all_model_summaries = []

for model_name, paths in MODEL_PATHS.items():
    print("=" * 80)
    print(f"F2 threshold tuning: {model_name}")
    print("=" * 80)

    model_output_dir = OUTPUT_DIR / model_name
    model_output_dir.mkdir(parents=True, exist_ok=True)

    dev_dataset, columns = load_dev_dataset(paths["dev_data_path"])
    print(f"Dev rows: {len(dev_dataset)}")
    print(f"Columns used: {columns}")

    probs, labels, avg_loss = run_forward_pass(
        paths["model_to_import_path"],
        dev_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )
    print(f"Forward pass complete. probs={probs.shape}, labels={labels.shape}, loss={avg_loss:.4f}")

    baseline_thresholds = np.full(len(LABEL_COLS), BASELINE_THRESHOLD, dtype=float)
    baseline_preds, baseline_metrics = evaluate_threshold_vector(
        probs,
        labels,
        baseline_thresholds,
        loss=avg_loss,
    )

    sweep_df, best_df, tuned_thresholds = tune_thresholds(probs, labels, THRESHOLD_GRID)
    tuned_preds, tuned_metrics = evaluate_threshold_vector(
        probs,
        labels,
        tuned_thresholds,
        loss=avg_loss,
    )

    sweep_path = model_output_dir / "threshold_sweep.csv"
    json_path = model_output_dir / "best_thresholds.json"
    sweep_df.to_csv(sweep_path, index=False)

    baseline_report = classification_report(
        labels,
        baseline_preds,
        target_names=LABEL_COLS,
        zero_division=0,
        output_dict=True,
    )
    tuned_report = classification_report(
        labels,
        tuned_preds,
        target_names=LABEL_COLS,
        zero_division=0,
        output_dict=True,
    )

    payload = {
        "model_name": model_name,
        "model_path": paths["model_to_import_path"],
        "dev_data_path": paths["dev_data_path"],
        "label_columns": LABEL_COLS,
        "threshold_grid": THRESHOLD_GRID.tolist(),
        "selection_metric": "per_label_f2",
        "selection_tie_breakers": ["recall", "precision", "closest_to_0.50"],
        "baseline_threshold": BASELINE_THRESHOLD,
        "baseline_metrics": baseline_metrics,
        "best_thresholds": {
            row["label"]: float(row["threshold"])
            for row in best_df.sort_values("label_idx").to_dict(orient="records")
        },
        "best_threshold_vector": tuned_thresholds.tolist(),
        "best_per_label_scores": best_df.sort_values("label_idx").to_dict(orient="records"),
        "tuned_metrics": tuned_metrics,
        "baseline_classification_report": baseline_report,
        "tuned_classification_report": tuned_report,
    }
    save_json(json_path, payload)

    summary = {
        "model_name": model_name,
        **{f"baseline_{k}": v for k, v in baseline_metrics.items()},
        **{f"tuned_{k}": v for k, v in tuned_metrics.items()},
        "sweep_path": str(sweep_path),
        "json_path": str(json_path),
    }
    all_model_summaries.append(summary)

    print("Best per-label thresholds:")
    display(best_df[["label", "threshold", "precision", "recall", "f1", "f2", "support", "predicted_positive"]])

    print("Baseline metrics at threshold 0.50:")
    display(pd.DataFrame([baseline_metrics]))

    print("Tuned metrics:")
    display(pd.DataFrame([tuned_metrics]))

    print(f"Saved sweep: {sweep_path}")
    print(f"Saved best thresholds: {json_path}")

    del dev_dataset, probs, labels, baseline_preds, tuned_preds, sweep_df, best_df
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(all_model_summaries)
summary_df


In [ ]:
summary_path = OUTPUT_DIR / "threshold_tuning_f2_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved summary: {summary_path}")
summary_df


We first fine-tuned the selected candidates (**DistilBERT, RoBERTa-base, and HateBERT**) for 4 epochs each. The best two candidates to proceed with were HateBERT and DistilBERT, with evaluation macro F1 scores of **0.8540** and **0.8445**, respectively.

After proceeding with threshold fine-tuning on the selected 4th-epoch models, we ended up with macro F1 scores of **0.8666** and **0.8786** for HateBERT and DistilBERT, respectively.

The following threshold values were selected for each model:

* **DistilBERT:** `[0.60, 0.40, 0.69, 0.68, 0.67, 0.48]`
* **HateBERT:** `[0.66, 0.53, 0.57, 0.69, 0.47, 0.55]`

Later, I decided to fine-tune the two selected models for two additional epochs using the same training techniques as before. This decision resulted in lower training and validation losses, along with better metrics on the evaluation set at the final training epoch.

The new macro F1 scores (without threshold fine-tuning yet) were **0.8888** and **0.8959** for HateBERT and DistilBERT, respectively.

After proceeding again with threshold fine-tuning on the selected 6th-epoch models, we ended up with macro F1 scores of **0.9011** and **0.9148** for HateBERT and DistilBERT, respectively.

The following threshold values were selected for each model:

* **DistilBERT:** `[0.74, 0.45, 0.74, 0.59, 0.61, 0.76]`
* **HateBERT:** `[0.58, 0.49, 0.76, 0.46, 0.76, 0.72]`

The final results are clearly in favor of DistilBERT, with the following macro metrics after the 6th epoch:

* **DistilBERT:**

  * Precision: **0.9131**
  * Recall: **0.9181**
  * F1: **0.9148**

* **HateBERT:**

  * Precision: **0.8879**
  * Recall: **0.9168**
  * F1: **0.9011**

The decision is to proceed with **DistilBERT as the main model** until further fine-tuning is performed or other models are evaluated. HateBERT will remain the secondary candidate for other production use cases, such as **A/B testing**.

The last remaining step is to test the model on unseen **test** data.